# Hands-on: choose one definition, predict the effect, then test it
**Yiwen Li · Fresh Eyes on CMIP · updated 9 November 2026**

This notebook runs independently from the seminar notebook. It uses the updated
1979–2024 ESMValTool calculation and starts each route from the same raw data.
**Default inputs are real ACCESS-CM2 historical + SSP5-8.5 output.**
Both source files must be available. The Binder build caches them; the prepared
Mac already has them. For an explicit software test only, choose
`WORKSHOP_MODE=synthetic`. There is no automatic fallback.

No Python knowledge is required. The setup, checks and plotting helpers are
folded; the CDO/NCO command cells and all outputs stay visible. Run folded cells
with **Shift+Enter** or **Run All**, and do not delete them. Click an ellipsis only
if you want to inspect the helper code.

## 02 · The calculation we are comparing

| Step | Definition used by every route |
|---|---|
| Input | ACCESS-CM2 monthly `tas` in Kelvin, historical + SSP5-8.5; or explicitly labeled synthetic test data |
| Main period | January 1979–December 2024: **552 months, 46 annual values** |
| Reference | January 1981–December 2010: 30 values for each calendar month |
| Monthly anomaly | Temperature minus the matching calendar-month reference field |
| Mean anomaly map | **Month-duration weighted** mean over the analysis period |
| Arctic mean | Latitude centres 65–90°N, all longitudes, **equal grid-point weights** |
| Annual stripes | **Equal weight for the 12 monthly samples** in each year |
| Comparison | Maximum absolute difference ≤ 0.0003°C, relative tolerance 0; coordinates and calendar labels agree |

The map and annual series deliberately use different time weights. This follows
`climate_statistics(period: full)` and `annual_statistics` in the updated
ESMValTool recipe. Replacing both with the same averaging operator changes the task.
The shared input check verifies matching grid, calendar and model/member metadata,
complete months, full calendar-month time bounds and finite input values.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, uuid
from datetime import datetime, timezone
from IPython.display import display, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "scripts/run_updated_workflows.sh").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts/run_updated_workflows.sh").exists(), "Open this notebook from the workshop repository."
os.chdir(ROOT)

# The seminar uses real ACCESS-CM2. Synthetic mode is an explicit software-test option.
DATA_MODE = os.environ.get("WORKSHOP_MODE", "real")  # "real" or "synthetic"
END_YEAR = int(os.environ.get("WORKSHOP_END_YEAR", "2024"))
assert DATA_MODE in {"synthetic", "real"}
assert END_YEAR >= 2024
os.environ["OMP_NUM_THREADS"] = "1"
if DATA_MODE == "synthetic":
    fixture = ROOT / "data/updated_fixture"
    subprocess.run([sys.executable, "scripts/generate_updated_fixture.py", str(fixture)], check=True)
    HIST, FUTURE = fixture / "historical.nc", fixture / "ssp585.nc"
else:
    directory = ROOT / "data/access_cm2"
    HIST = Path(os.environ.get("WORKSHOP_HISTORICAL", str(directory / "tas_Amon_ACCESS-CM2_historical_r1i1p1f1_gn_185001-201412.nc")))
    FUTURE = Path(os.environ.get("WORKSHOP_FUTURE", str(directory / "tas_Amon_ACCESS-CM2_ssp585_r1i1p1f1_gn_201501-210012.nc")))
    missing = [str(p) for p in (HIST, FUTURE) if not p.is_file()]
    if missing:
        raise FileNotFoundError("Real-data mode requires both raw files. No synthetic substitution is allowed. Missing: " + "; ".join(missing) + ". See docs/UPDATED_DEMO.md.")

RUN = ROOT / "outputs" / ("updated_" + DATA_MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN.mkdir(parents=True)
os.environ.update(DEMO_HIST=os.path.relpath(HIST, ROOT), DEMO_FUTURE=os.path.relpath(FUTURE, ROOT), DEMO_OUT=os.path.relpath(RUN, ROOT), DEMO_END=str(END_YEAR))
print("DATA MODE:", "SYNTHETIC SOFTWARE TEST — NOT ACCESS-CM2" if DATA_MODE == "synthetic" else "REAL ACCESS-CM2: historical + SSP5-8.5")
print("Analysis:", f"1979–{END_YEAR}", "| Monthly reference: 1981–2010")
print("Outputs:", RUN.relative_to(ROOT))

In [ ]:
subprocess.run([sys.executable, "scripts/check_updated_inputs.py", str(HIST), str(FUTURE), "--end-year", str(END_YEAR)], check=True)
for command in (["cdo", "-V"], ["ncks", "--version"]):
    result = subprocess.run(command, check=True, text=True, capture_output=True)
    print((result.stdout + result.stderr).splitlines()[0])
print("Python", sys.version.split()[0])

## 1 · Establish the reference result

Run all three independent implementations once. Keep their common baseline,
region and time weights fixed before exploring alternatives.

In [ ]:
%%bash
set -euo pipefail
bash scripts/run_updated_workflows.sh "$DEMO_HIST" "$DEMO_FUTURE" "$DEMO_OUT" "$DEMO_END"

In [ ]:
display(Image(filename=str(RUN / "updated_comparison.png")))

## 2 · Move the Arctic boundary to 70°N

Prediction: should narrowing the region change the annual anomaly? The answer
depends on the spatial structure of the data. These commands use each route's
own anomaly file. They retain equal grid-point weights and equal monthly weights.
Only the latitude threshold changes.

In [ ]:
%%bash
set -euo pipefail
mkdir -p "$DEMO_OUT/exercises"
cdo -O -L -s -b F64 -f nc sellonlatbox,0,360,70,90   "$DEMO_OUT/cdo/anomaly.nc" "$DEMO_OUT/exercises/cdo_arctic_70N.nc"
cdo -O -L -s -b F64 -f nc fldmean,weights=false   "$DEMO_OUT/exercises/cdo_arctic_70N.nc" "$DEMO_OUT/exercises/cdo_monthly_70N.nc"
ncwa -O -a lat,lon -d lat,70.0,90.0   "$DEMO_OUT/nco/anomaly.nc" "$DEMO_OUT/exercises/nco_monthly_70N.nc"

In [ ]:
import numpy as np
import xarray as xr
with xr.open_dataset(RUN / "exercises/cdo_monthly_70N.nc") as a, xr.open_dataset(RUN / "exercises/nco_monthly_70N.nc") as b:
    cdo70 = a.tas_anom.values.squeeze()
    nco70 = b.tas_anom.values.squeeze()
    np.testing.assert_allclose(cdo70, nco70, atol=0.0003, rtol=0)
    print("70°N CDO/NCO maximum difference:", float(abs(cdo70 - nco70).max()), "°C")
with xr.open_dataset(RUN / "cdo/arctic_monthly.nc") as original:
    print("70°N versus 65°N maximum difference:", float(abs(cdo70 - original.tas_anom.values.squeeze()).max()), "°C")

## 3 · Change the annual time weights only

`yearmean` gives equal weight to each monthly sample. `yearmonmean` weights
monthly values by the number of days in each month. The latter is a valid
alternative question, but it changes the annual definition in the shared recipe.
Keep the Arctic boundary at 65°N while testing this difference.

In [ ]:
%%bash
set -euo pipefail
cdo -O -s -b F64 -f nc yearmonmean "$DEMO_OUT/cdo/arctic_monthly.nc"   "$DEMO_OUT/exercises/cdo_annual_monthlength.nc"

In [ ]:
with xr.open_dataset(RUN / "exercises/cdo_annual_monthlength.nc") as weighted, xr.open_dataset(RUN / "cdo/arctic_annual.nc") as equal:
    difference = weighted.tas_anom.values.squeeze() - equal.tas_anom.values.squeeze()
    print("Annual month-length weighting minus equal-month weighting:")
    print("Maximum absolute change:", float(abs(difference).max()), "°C")
print("The shared recipe's annual result remains the original equal-month product.")

## 4 · Check the reference period residual

For each grid point and calendar month, the average anomaly over 1981–2010
should be close to zero. This checks the month matching, not whether all
1979–2024 anomalies average to zero.

In [ ]:
%%bash
set -euo pipefail
cdo -O -s -b F64 -f nc selyear,1981/2010 "$DEMO_OUT/cdo/anomaly.nc"   "$DEMO_OUT/exercises/baseline_anomaly.nc"
cdo -O -s -b F64 -f nc ymonmean "$DEMO_OUT/exercises/baseline_anomaly.nc"   "$DEMO_OUT/exercises/baseline_residual.nc"

In [ ]:
with xr.open_dataset(RUN / "exercises/baseline_residual.nc") as residual:
    maximum = float(abs(residual.tas_anom).max())
    assert maximum <= 0.0003
    print("PASS: maximum reference-period monthly residual =", maximum, "°C")

## Optional extension · Continue the scenario to 2100

Change `WORKSHOP_END_YEAR` (or `END_YEAR` in the setup cell) to `2100`, restart,
and run all cells. Use the same historical 1981–2010 reference, and retain
explicit historical/SSP5-8.5 labeling. This produces 122 annual values, 1979–2100.
It is an extension of the seminar calculation; do not compare its period-mean
map directly with the 1979–2024 reference map as if the definitions were equal.

Save your edited notebook, NetCDF outputs, validation report and figures before
closing a Binder session. All outputs from this run are under the printed path.